## 問題分析

### エッセンス
- `n`個の瓶に対して`m`回の範囲加算操作
- 最終的な平均値の床関数(floor)を求める

### 重要な洞察💡

**差分配列すら不要！**

平均を求めるには「総和」だけあればよい。
各操作`[a, b, v]`が総和に加える量は `v × (b - a + 1)` のみ。

```
総和 = Σ v_i × (b_i - a_i + 1)
平均 = floor(総和 / n)
```

## アルゴリズム比較表

| アプローチ | 時間計算量 | 空間計算量 | 可読性 | 備考 |
|---|---|---|---|---|
| **直接総和計算** | O(m) | O(1) | ★★★ | 最適解 |
| 差分配列 | O(n+m) | O(n) | ★★☆ | 過剰 |
| 愚直シミュレーション | O(n×m) | O(n) | ★★★ | TLE確定 |

---

## 実装

### 競技用（solve_competitive）

```python
def solve_competitive(n: int, operations: list[list[int]]) -> int:
    """
    競技プログラミング向け実装
    
    平均算出に差分配列は不要。
    各操作の寄与分を直接総和に加算するだけでO(m)で解ける。
    
    Time Complexity: O(m)
    Space Complexity: O(1)
    """
    total: int = sum(v * (b - a + 1) for a, b, v in operations)
    return total // n
```

### 業務用（solve_production）

```python
from typing import List


def solve_production(n: int, operations: List[List[int]]) -> int:
    """
    業務開発向け実装（型安全・エラーハンドリング重視）

    各操作 [a, b, v] が全体の総和に与える寄与は v*(b-a+1)。
    平均 = floor(Σ寄与 / n) で直接計算可能。

    Args:
        n: キャンディ瓶の数 (1 ≤ n ≤ 10^7)
        operations: [[a, b, v], ...] 形式の操作リスト
    Returns:
        全瓶の平均キャンディ数の床関数
    Raises:
        ValueError: 引数が制約を満たさない場合
    """
    if n <= 0:
        raise ValueError(f"n must be positive, got {n}")
    if not operations:
        return 0

    for op in operations:
        if len(op) != 3:
            raise ValueError(f"Each operation must have 3 elements, got {op}")
        a, b, v = op
        if not (1 <= a <= b <= n):
            raise ValueError(f"Invalid range [{a}, {b}] for n={n}")
        if v < 0:
            raise ValueError(f"v must be non-negative, got {v}")

    total: int = sum(v * (b - a + 1) for a, b, v in operations)
    return total // n
```

---

## 動作検証

```
# サンプル1
n=5, ops=[[1,2,100],[2,5,100],[3,4,100]]
寄与: 100×2 + 100×4 + 100×2 = 200+400+200 = 800
平均: 800 // 5 = 160 ✅

# 問題文Example
n=5, ops=[[1,3,10],[3,5,10]]
寄与: 10×3 + 10×3 = 30+30 = 60
実際は [1,3,10]→1-3番目の瓶, [3,5,10]→3-5番目の瓶とすると重複は3番目のみ
総和: 配列状態 [10, 10, 20, 10, 10] = 60
平均: 60 // 5 = 12
```

## HackerRank 提出コード

```python
def solve(n: int, operations: list[list[int]]) -> int:
    """
    Time Complexity: O(m)  ← 操作数のみに依存
    Space Complexity: O(1) ← 追加配列不要

    核心: floor(平均) = floor(総和/n)
    各操作[a,b,v]の総和寄与 = v * (b - a + 1)
    """
    total: int = sum(v * (b - a + 1) for a, b, v in operations)
    return total // n
```

### なぜ差分配列が不要か？

```
差分配列が必要なケース  → 各瓶の最終値を個別に知りたい時
今回必要なもの         → 総和だけ

総和 = Σ(全瓶の値)
    = Σ_i Σ_{op: i∈[a,b]} v
    = Σ_{op} v × (その操作で加算される瓶の数)
    = Σ_{op} v × (b - a + 1)   ← O(m)で完結！
```